In [1]:
%load_ext autoreload
%autoreload 2
import os

# I believe this environment variable should be set before importing t
os.environ["PYt_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch as t
from transformers import AutoModelForCausalLM, AutoTokenizer
import argparse
import itertools
import random
import json
import torch.multiprocessing as mp
import time
import huggingface_hub
from datasets import config
from transformers import AutoTokenizer
import demo_config
from tqdm import tqdm
import matplotlib.pyplot as plt

# Kind of janky double importing dictionary_learning.dictionary_learning, but it works
# This is leftover from when dictionary_learning was a only used as a submodule
from dictionary_learning.dictionary_learning.utils import (
    hf_dataset_to_generator,
    hf_mixed_dataset_to_generator,
    hf_sequence_packing_dataset_to_generator,
)
from dictionary_learning.dictionary_learning.pytorch_buffer import ActivationBuffer
from dictionary_learning.dictionary_learning.evaluation import evaluate
from dictionary_learning.dictionary_learning.training import trainSAE
import dictionary_learning.dictionary_learning.utils as utils


if os.path.exists('/u/eboix/moe_distillation/sae_demo'):
    os.chdir('/u/eboix/moe_distillation/sae_demo')

In [2]:
sae_dir = '._saes_EleutherAI_pythia-70m-deduped_top_k/mlp_in_3/trainer_0/'
device = t.device('cuda' if t.cuda.is_available() else 'cpu')
n_test_batches = 100

In [3]:
# load config.json from sae_dir
config_path = os.path.join(sae_dir, 'config.json')
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        config_data = json.load(f)
else:
    assert(False), f'Config file not found at {config_path}'

model_name = config_data['trainer']['lm_name']
layer_idx = config_data['trainer']['layer']
io = config_data['buffer']['io']
dtype = t.float32

# load autoencoder from sae_file
from dictionary_learning.dictionary_learning.trainers.top_k import AutoEncoderTopK
ae_file = os.path.join(sae_dir, 'ae.pt')
autoencoder = AutoEncoderTopK.from_pretrained(ae_file)
autoencoder = autoencoder.to(device)
autoencoder.eval()
print('Autoencoder loaded from:', ae_file)

model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="auto", torch_dtype=dtype
)

model = utils.truncate_model(model, layer_idx)
model.eval()
model = model.to(device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
submodule = utils.get_submodule(model, layer_idx).mlp
submodule = submodule.to(device)
print('Model loaded:', model_name)

generator = hf_dataset_to_generator("monology/pile-uncopyrighted", 
    data_files={"validation": "val.jsonl.zst"},
    split="validation",)

activation_dim = config_data["trainer"]["activation_dim"]

activation_buffer = ActivationBuffer(
    generator,
    model,
    submodule,
    n_ctxs=config_data['buffer']['n_ctxs'],
    ctx_len=config_data['buffer']['ctx_len'],
    refresh_batch_size=config_data['buffer']['refresh_batch_size'],
    out_batch_size=config_data['buffer']['out_batch_size'],
    io=io,
    d_submodule=activation_dim,
    device=device,
)

print('Initialized activation buffer')

Autoencoder loaded from: ._saes_EleutherAI_pythia-70m-deduped_top_k/mlp_in_3/trainer_0/ae.pt
Model parameters before truncation: 70,426,624
Model parameters after truncation: 38,366,208
Model loaded: EleutherAI/pythia-70m-deduped
Initialized activation buffer


In [4]:
# Estimate variance of y_teacher
# first estimate mean
n_estimate_iters = 4
y_mean = t.zeros(activation_dim, device=device)
for _ in tqdm(range(n_estimate_iters)):
    x = next(activation_buffer)
    x = x.to(device)
    y_teacher = submodule(x)
    y_mean += y_teacher.mean(dim=0)
y_mean /= n_estimate_iters

y_variance = 0.0
for _ in tqdm(range(n_estimate_iters)):
    x = next(activation_buffer)
    x = x.to(device)
    y_teacher = submodule(x)
    y_variance += t.mean(t.mean((y_teacher - y_mean) ** 2, dim=0))
y_variance /= n_estimate_iters
y_variance = y_variance.item()
print('Variance in y', y_variance)


  0%|                                                                                                              | 0/4 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 265.03it/s]

Variance in y 0.17905718088150024


In [5]:
# MLP class
class MLP(t.nn.Module):
    def __init__(self, d_in, d_out, d_hidden):
        super().__init__()
        self.linear1 = t.nn.Linear(d_in, d_hidden)
        self.activation = t.nn.GELU()
        self.linear2 = t.nn.Linear(d_hidden, d_out)

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        return x

In [8]:
from tqdm.notebook import tqdm
import math

# train NaiveMoeLowRankModel as student model to learn submodule

device = activation_buffer.device
d = activation_buffer.d_submodule
d_hidden = d*4
# d_hidden = d*6

# m=16384
k=40
num_tokens = 50_000_000
batch_size = activation_buffer.out_batch_size
num_batches = num_tokens // batch_size

# student_model = ParallelMoEWithRotationNaive(d, d_hidden, d_expert=1, m=m, k=k).to(device)
# student_model =  MLP(d, d, d_hidden).to(device)
student_model = t.nn.Linear(d,d).to(device)
# student_model.linear2.bias.data = submodule.dense_4h_to_h.bias.data.clone()
# Create parameter groups with different learning rates
optimizer = t.optim.AdamW([
    {'params': student_model.parameters(), 'lr': 1e-2}
])
# Set up a cosine learning rate scheduler
scheduler = t.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_batches, eta_min=1e-5)
# Training loop
criterion = t.nn.MSELoss()
losses = []

progress_bar = tqdm(range(num_batches), desc="Training Student Model")

for i in progress_bar:
    # Get a batch of input activations
    x_batch = next(activation_buffer).to(device)
    
    # Get the target outputs by passing through the original submodule
    with t.no_grad():
        target_output = submodule(x_batch)

    with t.no_grad():
        x_recon = autoencoder(x_batch)
        x_err = x_batch - x_recon
        y_recon = submodule(x_recon)
    
    # Forward pass through student model
    student_output = student_model(x_err)
    student_output = student_output + y_recon
        
    # Compute loss
    loss = criterion(student_output, target_output)
    variance_explained = 1 - (loss.item() / y_variance)
    progress_bar.set_postfix({"loss": f"{loss.item():.6f}", "variance_explained": f"{variance_explained:.4f}", 'lr': f"{scheduler.get_last_lr()[0]:.6f}"})
    
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

Training Student Model:   0%|          | 0/24414 [00:00<?, ?it/s]

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f665bb70c20>>
Traceback (most recent call last):
  File "/u/eboix/miniconda3/envs/pytorch_env2/lib/python3.13/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


KeyboardInterrupt: 

In [33]:
from tqdm.notebook import tqdm
import math

# train NaiveMoeLowRankModel as student model to learn submodule

device = activation_buffer.device
d = activation_buffer.d_submodule
d_hidden = d*4
# d_hidden = d*6

# m=16384
k=40
num_tokens = 50_000_000
batch_size = activation_buffer.out_batch_size
num_batches = num_tokens // batch_size

# student_model = ParallelMoEWithRotationNaive(d, d_hidden, d_expert=1, m=m, k=k).to(device)
student_model =  MLP(d, d, d_hidden).to(device)
# student_model = t.nn.Linear(d,d).to(device)
# student_model.linear2.bias.data = submodule.dense_4h_to_h.bias.data.clone()
# Create parameter groups with different learning rates
optimizer = t.optim.AdamW([
    {'params': student_model.parameters(), 'lr': 3e-4}
])
# Set up a cosine learning rate scheduler
scheduler = t.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_batches, eta_min=1e-5)
# Training loop
criterion = t.nn.MSELoss()
losses = []

progress_bar = tqdm(range(num_batches), desc="Training Student Model")

for i in progress_bar:
    # Get a batch of input activations
    x_batch = next(activation_buffer).to(device)
    
    # Get the target outputs by passing through the original submodule
    with t.no_grad():
        target_output = submodule(x_batch)

    with t.no_grad():
        x_recon = autoencoder(x_batch)
        x_err = x_batch - x_recon
        y_recon = submodule(x_recon)
    
    # Forward pass through student model
    student_output = student_model(x_err)
    student_output = student_output + y_recon
        
    # Compute loss
    loss = criterion(student_output, target_output)
    variance_explained = 1 - (loss.item() / y_variance)
    progress_bar.set_postfix({"loss": f"{loss.item():.6f}", "variance_explained": f"{variance_explained:.4f}", 'lr': f"{scheduler.get_last_lr()[0]:.6f}"})
    
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

Training Student Model:   0%|          | 0/24414 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [34]:
from tqdm.notebook import tqdm
import math

# train NaiveMoeLowRankModel as student model to learn submodule

device = activation_buffer.device
d = activation_buffer.d_submodule
d_hidden = d*4
# d_hidden = d*6

# m=16384
k=40
num_tokens = 50_000_000
batch_size = activation_buffer.out_batch_size
num_batches = num_tokens // batch_size

# student_model = ParallelMoEWithRotationNaive(d, d_hidden, d_expert=1, m=m, k=k).to(device)
student_model =  MLP(d, d, d_hidden).to(device)
student_model2 =  MLP(d, d, d_hidden).to(device)
# student_model = t.nn.Linear(d,d).to(device)
# student_model.linear2.bias.data = submodule.dense_4h_to_h.bias.data.clone()
# Create parameter groups with different learning rates
optimizer = t.optim.AdamW([
    {'params': student_model.parameters(), 'lr': 3e-4},
    {'params': student_model2.parameters(), 'lr': 3e-4}
])
# Set up a cosine learning rate scheduler
scheduler = t.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_batches, eta_min=1e-5)
# Training loop
criterion = t.nn.MSELoss()
losses = []

progress_bar = tqdm(range(num_batches), desc="Training Student Model")

for i in progress_bar:
    # Get a batch of input activations
    x_batch = next(activation_buffer).to(device)
    
    # Get the target outputs by passing through the original submodule
    with t.no_grad():
        target_output = submodule(x_batch)

    with t.no_grad():
        x_recon = autoencoder(x_batch)
        x_err = x_batch - x_recon
        # y_recon = submodule(x_recon)
    
    # Forward pass through student model
    student_output = student_model(x_err) + student_model2(x_recon)
        
    # Compute loss
    loss = criterion(student_output, target_output)
    variance_explained = 1 - (loss.item() / y_variance)
    progress_bar.set_postfix({"loss": f"{loss.item():.6f}", "variance_explained": f"{variance_explained:.4f}", 'lr': f"{scheduler.get_last_lr()[0]:.6f}"})
    
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

Training Student Model:   0%|          | 0/24414 [00:00<?, ?it/s]

StopIteration: End of data stream reached

In [6]:
from tqdm.notebook import tqdm
import math

# train NaiveMoeLowRankModel as student model to learn submodule

device = activation_buffer.device
d = activation_buffer.d_submodule
d_hidden = d*4
# d_hidden = d*6

# m=16384
k=40
num_tokens = 50_000_000
batch_size = activation_buffer.out_batch_size
num_batches = num_tokens // batch_size

# student_model = ParallelMoEWithRotationNaive(d, d_hidden, d_expert=1, m=m, k=k).to(device)
student_model =  t.nn.Linear(d,d).to(device)
student_model2 =  MLP(d, d, 2*d_hidden).to(device)
# student_model = t.nn.Linear(d,d).to(device)
# student_model.linear2.bias.data = submodule.dense_4h_to_h.bias.data.clone()
# Create parameter groups with different learning rates
optimizer = t.optim.AdamW([
    {'params': student_model.parameters(), 'lr': 3e-4},
    {'params': student_model2.parameters(), 'lr': 3e-4}
])
# Set up a cosine learning rate scheduler
scheduler = t.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_batches, eta_min=1e-5)
# Training loop
criterion = t.nn.MSELoss()
losses = []

progress_bar = tqdm(range(num_batches), desc="Training Student Model")

for i in progress_bar:
    # Get a batch of input activations
    x_batch = next(activation_buffer).to(device)
    
    # Get the target outputs by passing through the original submodule
    with t.no_grad():
        target_output = submodule(x_batch)

    with t.no_grad():
        x_recon = autoencoder(x_batch)
        x_err = x_batch - x_recon
        # y_recon = submodule(x_recon)
    
    # Forward pass through student model
    student_output = student_model(x_err) + student_model2(x_recon)
        
    # Compute loss
    loss = criterion(student_output, target_output)
    variance_explained = 1 - (loss.item() / y_variance)
    progress_bar.set_postfix({"loss": f"{loss.item():.6f}", "variance_explained": f"{variance_explained:.4f}", 'lr': f"{scheduler.get_last_lr()[0]:.6f}"})
    
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

Training Student Model:   0%|          | 0/24414 [00:00<?, ?it/s]

In [ ]:
from tqdm.notebook import tqdm
import math

# train NaiveMoeLowRankModel as student model to learn submodule

device = activation_buffer.device
d = activation_buffer.d_submodule
d_hidden = d*4
# d_hidden = d*6

# m=16384
k=40
num_tokens = 50_000_000
batch_size = activation_buffer.out_batch_size
num_batches = num_tokens // batch_size

# student_model = ParallelMoEWithRotationNaive(d, d_hidden, d_expert=1, m=m, k=k).to(device)
student_model =  t.nn.Linear(d,d).to(device)
student_model2 =  MLP(d, d, d_hidden).to(device)
student_model2.linear1.weight.data = submodule.dense_h_to_4h.weight.data.clone()
student_model2.linear1.bias.data = submodule.dense_h_to_4h.bias.data.clone()
student_model2.linear2.weight.data = submodule.dense_4h_to_h.weight.data.clone()
student_model2.linear2.bias.data = submodule.dense_4h_to_h.bias.data.clone()
# student_model =  MLP(d, d, d_hidden).to(device)
# student_model = t.nn.Linear(d,d).to(device)
# student_model.linear2.bias.data = submodule.dense_4h_to_h.bias.data.clone()
# Create parameter groups with different learning rates
optimizer = t.optim.AdamW([
    {'params': student_model.parameters(), 'lr': 3e-4},
    {'params': student_model2.parameters(), 'lr': 3e-4}
])
# Set up a cosine learning rate scheduler
scheduler = t.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_batches, eta_min=1e-5)
# Training loop
criterion = t.nn.MSELoss()
losses = []

progress_bar = tqdm(range(num_batches), desc="Training Student Model")

for i in progress_bar:
    # Get a batch of input activations
    x_batch = next(activation_buffer).to(device)
    
    # Get the target outputs by passing through the original submodule
    with t.no_grad():
        target_output = submodule(x_batch)

    with t.no_grad():
        x_recon = autoencoder(x_batch)
        x_err = x_batch - x_recon
        # y_recon = submodule(x_recon)
    
    # Forward pass through student model
    student_output = student_model(x_err) + student_model2(x_recon)
        
    # Compute loss
    loss = criterion(student_output, target_output)
    variance_explained = 1 - (loss.item() / y_variance)
    progress_bar.set_postfix({"loss": f"{loss.item():.6f}", "variance_explained": f"{variance_explained:.4f}", 'lr': f"{scheduler.get_last_lr()[0]:.6f}"})
    
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

Training Student Model:   0%|          | 0/24414 [00:00<?, ?it/s]

In [ ]:
from tqdm.notebook import tqdm
import math

# train NaiveMoeLowRankModel as student model to learn submodule

device = activation_buffer.device
d = activation_buffer.d_submodule
d_hidden = d*4
# d_hidden = d*6

# m=16384
k=40
num_tokens = 50_000_000
batch_size = activation_buffer.out_batch_size
num_batches = num_tokens // batch_size

# student_model = ParallelMoEWithRotationNaive(d, d_hidden, d_expert=1, m=m, k=k).to(device)
student_model =  MLP(d,d, d_hidden).to(device)
student_model2 =  MLP(d, d, d_hidden).to(device)
student_model2.linear1.weight.data = submodule.dense_h_to_4h.weight.data.clone()
student_model2.linear1.bias.data = submodule.dense_h_to_4h.bias.data.clone()
student_model2.linear2.weight.data = submodule.dense_4h_to_h.weight.data.clone()
student_model2.linear2.bias.data = submodule.dense_4h_to_h.bias.data.clone()
# student_model =  MLP(d, d, d_hidden).to(device)
# student_model = t.nn.Linear(d,d).to(device)
# student_model.linear2.bias.data = submodule.dense_4h_to_h.bias.data.clone()
# Create parameter groups with different learning rates
optimizer = t.optim.AdamW([
    {'params': student_model.parameters(), 'lr': 3e-4},
    {'params': student_model2.parameters(), 'lr': 3e-4}
])
# Set up a cosine learning rate scheduler
scheduler = t.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_batches, eta_min=1e-5)
# Training loop
criterion = t.nn.MSELoss()
losses = []

progress_bar = tqdm(range(num_batches), desc="Training Student Model")

for i in progress_bar:
    # Get a batch of input activations
    x_batch = next(activation_buffer).to(device)
    
    # Get the target outputs by passing through the original submodule
    with t.no_grad():
        target_output = submodule(x_batch)

    with t.no_grad():
        x_recon = autoencoder(x_batch)
        x_err = x_batch - x_recon
        # y_recon = submodule(x_recon)
    
    # Forward pass through student model
    student_output = student_model(x_err) + student_model2(x_recon)
        
    # Compute loss
    loss = criterion(student_output, target_output)
    variance_explained = 1 - (loss.item() / y_variance)
    progress_bar.set_postfix({"loss": f"{loss.item():.6f}", "variance_explained": f"{variance_explained:.4f}", 'lr': f"{scheduler.get_last_lr()[0]:.6f}"})
    
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

In [27]:
from tqdm.notebook import tqdm
import math

# train NaiveMoeLowRankModel as student model to learn submodule

device = activation_buffer.device
d = activation_buffer.d_submodule
d_hidden = d*4
# d_hidden = d*6

# m=16384
k=40
num_tokens = 50_000_000
batch_size = activation_buffer.out_batch_size
num_batches = num_tokens // batch_size

# student_model = ParallelMoEWithRotationNaive(d, d_hidden, d_expert=1, m=m, k=k).to(device)
# student_model =  MLP(d, d, d_hidden).to(device)
student_model = t.nn.Linear(d,d).to(device)
# student_model.linear2.bias.data = submodule.dense_4h_to_h.bias.data.clone()
# Create parameter groups with different learning rates
optimizer = t.optim.AdamW([
    {'params': student_model.parameters(), 'lr': 1e-2}
])
# Set up a cosine learning rate scheduler
scheduler = t.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_batches, eta_min=1e-5)
# Training loop
criterion = t.nn.MSELoss()
losses = []

progress_bar = tqdm(range(num_batches), desc="Training Student Model")

for i in progress_bar:
    # Get a batch of input activations
    x_batch = next(activation_buffer).to(device)
    
    # Get the target outputs by passing through the original submodule
    with t.no_grad():
        target_output = submodule(x_batch)

    with t.no_grad():
        x_recon = autoencoder(x_batch)
        x_err = x_batch - x_recon
        y_err = submodule(x_err)
    
    # Forward pass through student model
    student_output = student_model(x_recon)
    student_output = student_output + y_err
        
    # Compute loss
    loss = criterion(student_output, target_output)
    variance_explained = 1 - (loss.item() / y_variance)
    progress_bar.set_postfix({"loss": f"{loss.item():.6f}", "variance_explained": f"{variance_explained:.4f}", 'lr': f"{scheduler.get_last_lr()[0]:.6f}"})
    
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

Training Student Model:   0%|          | 0/24414 [00:00<?, ?it/s]

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f665bb70c20>>
Traceback (most recent call last):
  File "/u/eboix/miniconda3/envs/pytorch_env2/lib/python3.13/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


KeyboardInterrupt: 

In [26]:
from tqdm.notebook import tqdm
import math

# train NaiveMoeLowRankModel as student model to learn submodule

device = activation_buffer.device
d = activation_buffer.d_submodule
d_hidden = d*4
# d_hidden = d*6

# m=16384
k=40
num_tokens = 50_000_000
batch_size = activation_buffer.out_batch_size
num_batches = num_tokens // batch_size

# student_model = ParallelMoEWithRotationNaive(d, d_hidden, d_expert=1, m=m, k=k).to(device)
# student_model =  MLP(d, d, d_hidden).to(device)
student_model = t.nn.Linear(d,d).to(device)
student_model2 = t.nn.Linear(d,d).to(device)
# student_model.linear2.bias.data = submodule.dense_4h_to_h.bias.data.clone()
# Create parameter groups with different learning rates
optimizer = t.optim.AdamW([
    {'params': student_model.parameters(), 'lr': 1e-3},
    {'params': student_model2.parameters(), 'lr': 1e-3}
])
# Set up a cosine learning rate scheduler
scheduler = t.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_batches, eta_min=1e-5)
# Training loop
criterion = t.nn.MSELoss()
losses = []

progress_bar = tqdm(range(num_batches), desc="Training Student Model")

for i in progress_bar:
    # Get a batch of input activations
    x_batch = next(activation_buffer).to(device)
    
    # Get the target outputs by passing through the original submodule
    with t.no_grad():
        target_output = submodule(x_batch)

    with t.no_grad():
        x_recon = autoencoder(x_batch)
        x_err = x_batch - x_recon
        err_magnitudes = t.sum(x_err ** 2, dim=1).sqrt()
        z = t.randn_like(x_recon)
        z_magnitudes = t.sum(z ** 2, dim=1).sqrt()
        z = z / z_magnitudes.unsqueeze(1) # unit vectors
        z_proj = z - (t.sum(x_recon * z, dim=1) / t.sqrt(t.sum(x_recon ** 2, dim=1))).unsqueeze(1) * x_recon

        z = z_proj * err_magnitudes.unsqueeze(1)
#        * t.std(x_recon, dim=1, keepdim=True)
        y_recon = submodule(x_recon+z)
    
    # Forward pass through student model
    student_output = student_model(x_err) + student_model2(z)
    student_output = student_output + y_recon
        
    # Compute loss
    loss = criterion(student_output, target_output)
    variance_explained = 1 - (loss.item() / y_variance)
    progress_bar.set_postfix({"loss": f"{loss.item():.6f}", "variance_explained": f"{variance_explained:.4f}", 'lr': f"{scheduler.get_last_lr()[0]:.6f}"})
    
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

Training Student Model:   0%|          | 0/24414 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [21]:
t.sum(z**2)

tensor(1503901., device='cuda:0')

In [20]:
t.std(x_err,dim=1)

tensor([0.3433, 0.2822, 0.4234,  ..., 0.2499, 0.3357, 0.2743], device='cuda:0')